In [1]:
%%bash
# --- STEP 1: PREP HOST ---
echo "⚙️ Installing QEMU Static Emulators..."
apt-get -qq update
apt-get -qq install -y qemu-user-static

# --- STEP 2: FETCH ARM FILESYSTEM ---
# We use Ubuntu 22.04 ARM64 Base (Compatible with Pi 4 setups)
echo "⬇️ Downloading Ubuntu ARM64 Base Image..."
mkdir -p /content/arm_env
wget -q http://cdimage.ubuntu.com/ubuntu-base/releases/22.04/release/ubuntu-base-22.04-base-arm64.tar.gz -O /content/base.tar.gz

# --- STEP 3: EXTRACT & CONFIGURE ---
echo "📂 Extracting Filesystem (This takes a moment)..."
tar -xzf /content/base.tar.gz -C /content/arm_env

# Copy the QEMU binary INTO the jail so it can translate inside
cp /usr/bin/qemu-aarch64-static /content/arm_env/usr/bin/

# Copy DNS config so you have Internet inside the ARM env
cp /etc/resolv.conf /content/arm_env/etc/

# --- STEP 4: CREATE LAUNCHER SCRIPT ---
# This script handles the complex mounting of system folders
cat <<EOF > /content/enter_arm_env.sh
#!/bin/bash
# Mount critical system directories
mount -t proc /proc /content/arm_env/proc
mount -t sysfs /sys /content/arm_env/sys
mount -o bind /dev /content/arm_env/dev
mount -o bind /dev/pts /content/arm_env/dev/pts

# Enter the ARM Environment
echo "🚀 Entering ARM64 Environment..."
echo "💡 Type 'exit' to return to standard Colab."
chroot /content/arm_env /bin/bash
EOF

chmod +x /content/enter_arm_env.sh
echo "✅ Setup Complete. Run '!./enter_arm_env.sh' to start."

⚙️ Installing QEMU Static Emulators...
Selecting previously unselected package binfmt-support.
(Reading database ... 121852 files and directories currently installed.)
Preparing to unpack .../binfmt-support_2.2.1-2_amd64.deb ...
Unpacking binfmt-support (2.2.1-2) ...
Selecting previously unselected package qemu-user-static.
Preparing to unpack .../qemu-user-static_1%3a6.2+dfsg-2ubuntu6.27_amd64.deb ...
Unpacking qemu-user-static (1:6.2+dfsg-2ubuntu6.27) ...
Setting up qemu-user-static (1:6.2+dfsg-2ubuntu6.27) ...
Setting up binfmt-support (2.2.1-2) ...
invoke-rc.d: could not determine current runlevel
invoke-rc.d: policy-rc.d denied execution of restart.
Created symlink /etc/systemd/system/multi-user.target.wants/binfmt-support.service → /lib/systemd/system/binfmt-support.service.
Processing triggers for man-db (2.10.2-1) ...
⬇️ Downloading Ubuntu ARM64 Base Image...
📂 Extracting Filesystem (This takes a moment)...
✅ Setup Complete. Run '!./enter_arm_env.sh' to start.


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [2]:
%%bash
# --- FIX 1: Force Register QEMU Interpreters ---
# Since the service failed to start automatically, we manually register the formats.
echo "🔧 Forcing binfmt registration..."
update-binfmts --enable qemu-aarch64

# Verify it worked - you should see 'enabled'
if grep -q "qemu-aarch64" /proc/sys/fs/binfmt_misc/status; then
    echo "✅ QEMU-AARCH64 is now ACTIVE."
else
    echo "⚠️ Warning: Registration might have failed. Attempting alternate method..."
    # Fallback registration attempt
    service binfmt-support start
fi

# --- FIX 2: Create a Robust Launcher ---
# We make the mounting less strict to avoid the 'read-only' error
cat <<EOF > /content/enter_arm_env.sh
#!/bin/bash

# 1. Mount proc (Essential for apt/process lists)
# We check if it's already mounted to avoid errors
if ! mountpoint -q /content/arm_env/proc; then
    mount -t proc proc /content/arm_env/proc || echo "⚠️ Could not mount /proc (Ignore if shell works)"
fi

# 2. Bind /dev (Essential for random/null)
if ! mountpoint -q /content/arm_env/dev; then
    mount --bind /dev /content/arm_env/dev
fi

# 3. Enter the environment
echo "🚀 Entering ARM64 Environment..."
# We explicitly call the qemu loader just in case, though chroot should now work
chroot /content/arm_env /bin/bash
EOF

chmod +x /content/enter_arm_env.sh
echo "✅ Launcher patched."

🔧 Forcing binfmt registration...
⚠️ Warning: Registration might have failed. Attempting alternate method...
 * Enabling additional executable binary formats binfmt-support
   ...done.
✅ Launcher patched.


In [3]:
!./enter_arm_env.sh

mount: /content/arm_env/proc: cannot mount proc read-only.
⚠️ Could not mount /proc (Ignore if shell works)
🚀 Entering ARM64 Environment...
bash: cannot set terminal process group (188): Inappropriate ioctl for device
bash: no job control in this shell
bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)
]0;root@0c9f3fdd5c24: /root@0c9f3fdd5c24:/# exit
exit


In [4]:
from google.colab import drive
import os

# 1. Mount Google Drive to the Colab Host
print("🔄 Mounting Google Drive...")
drive.mount('/content/drive')

# 2. Create your Workspace Folder
# This folder will appear on your Windows Laptop as "G:\My Drive\ARM_Workspace"
workspace_path = "/content/drive/MyDrive/ARM_Workspace"
if not os.path.exists(workspace_path):
    os.makedirs(workspace_path)
    print(f"✅ Created new folder: {workspace_path}")
else:
    print(f"📂 Found existing folder: {workspace_path}")

# 3. Create the 'Bind' Script (Bash)
# We rewrite the launcher to map your Drive folder to '/root/workspace' inside the ARM machine
setup_script = f"""#!/bin/bash
# Re-mount system folders just in case
mount -t proc proc /content/arm_env/proc 2>/dev/null
mount --bind /dev /content/arm_env/dev 2>/dev/null
mount --bind /sys /content/arm_env/sys 2>/dev/null

# --- THE CRITICAL PART ---
# Create the mount point inside the jail
mkdir -p /content/arm_env/root/workspace

# Bind the Google Drive folder into the jail
mount --bind "{workspace_path}" /content/arm_env/root/workspace

echo "🔗 Google Drive connected to: /root/workspace"
echo "🚀 Entering ARM64 Environment..."

# Enter the jail
chroot /content/arm_env /bin/bash
"""

with open("/content/enter_arm_env.sh", "w") as f:
    f.write(setup_script)

!chmod +x /content/enter_arm_env.sh
print("✅ Bridge Built. Run '!./enter_arm_env.sh' to start.")

🔄 Mounting Google Drive...
Mounted at /content/drive
📂 Found existing folder: /content/drive/MyDrive/ARM_Workspace
✅ Bridge Built. Run '!./enter_arm_env.sh' to start.


In [5]:
!./enter_arm_env.sh

🔗 Google Drive connected to: /root/workspace
🚀 Entering ARM64 Environment...
bash: cannot set terminal process group (188): Inappropriate ioctl for device
bash: no job control in this shell
bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)
]0;root@0c9f3fdd5c24: /root@0c9f3fdd5c24:/# exit
exit
